In [1]:
# Optimizing k-NN performance: k selection, normalization, and PCA

import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# ---
# 1) Load dataset (moderately high-dimensional)
# ---
data = load_breast_cancer()
X, y = data.data, data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
print("Original dimensions:", X_train.shape)

# ---
# 2) Normalize features (very important for k-NN)
# ---

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

Original dimensions: (426, 30)


In [6]:
# 3) Choose optimal k (simple search)
# ---
best_k = None
best_acc = 0

for k in range(1, 21):
    knn = KNeighborsClassifier(n_neighbors=k, metric="euclidean",n_jobs=-1)
    knn.fit(X_train_scaled, y_train)
    y_pred = knn.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)
    print(f"With k :{k}, accuracy: {acc:.4f}")

    if acc > best_acc:
        best_acc = acc
        best_k = k

print(f"Best k found: {best_k} with accuracy: {best_acc:.4f}")

With k :1, accuracy: 0.9510
With k :2, accuracy: 0.9371
With k :3, accuracy: 0.9720
With k :4, accuracy: 0.9580
With k :5, accuracy: 0.9790
With k :6, accuracy: 0.9720
With k :7, accuracy: 0.9790
With k :8, accuracy: 0.9720
With k :9, accuracy: 0.9650
With k :10, accuracy: 0.9650
With k :11, accuracy: 0.9720
With k :12, accuracy: 0.9790
With k :13, accuracy: 0.9580
With k :14, accuracy: 0.9650
With k :15, accuracy: 0.9580
With k :16, accuracy: 0.9580
With k :17, accuracy: 0.9510
With k :18, accuracy: 0.9510
With k :19, accuracy: 0.9510
With k :20, accuracy: 0.9510
Best k found: 5 with accuracy: 0.9790


In [5]:
# 4) Dimensionality Reduction with PCA
# ---
pca = PCA(n_components=10)    # reduce dimensions
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

print("Reduced dimensions:", X_train_pca.shape)

# Train k-NN after PCA using the best k
knn_pca = KNeighborsClassifier(n_neighbors=best_k)
knn_pca.fit(X_train_pca, y_train)

y_pred_pca = knn_pca.predict(X_test_pca)
acc_pca = accuracy_score(y_test, y_pred_pca)

print(f"Accuracy after PCA: {acc_pca:.4f}")
print("Explained variance by 10 components:", 
    round(np.sum(pca.explained_variance_ratio_), 3))

Reduced dimensions: (426, 10)
Accuracy after PCA: 0.9790
Explained variance by 10 components: 0.954
